In [ ]:
import os
from pathlib import Path

import autoroot
import hydra
import matplotlib.pyplot as plt
import torch

from src.functionals.vandermonde import *
from src.modules.lit_module import LitModule
from src.utils import initialize_resolvers

initialize_resolvers()


def data_illustration(inputs, outputs, rho=None, dim=0):

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    ax[0].plot(inputs[0, :, dim].numpy())
    ax[0].plot(outputs[0, :, 0].numpy())

    if rho is not None:
        ax[1].plot(rho[dim])

In [ ]:
path = "./logs/train/2024-05-17_10-33-27"
# path = "./logs/train/airy"
# path = "./logs/train/rand"
# path = "./logs/train/exp"
# path = "./logs/train/step"


path = Path(path)
ckpt = max((path / "checkpoints").glob("*.ckpt"), key=os.path.getctime)
ckpt_init = path / "checkpoints" / "initial_checkpoint.ckpt"
model_init, model = LitModule.load_from_checkpoint(ckpt_init), LitModule.load_from_checkpoint(ckpt)
ssm_init, ssm = model_init.model.layers[0], model.model.layers[0]
lf = hydra.utils.instantiate(model.hparams.cfg.data).lf
seq_length, D, dt = lf.T, lf.D, lf.dt
print(lf)
print(ckpt.name)

### Target

In [ ]:
ssm

## Model

In [ ]:
for p in ssm.parameters():
    print(torch.norm(p))

In [ ]:
import matplotlib.pyplot as plt

a = torch.empty(100, 100)

torch.nn.init.xavier_uniform_(a)
a = a.numpy() * 0.9

In [ ]:
plt.scatter(np.linalg.eigvals(a).real, np.linalg.eigvals(a).imag)

In [ ]:
import numpy as np

e1 = np.linalg.eigvals(ssm.rnn_layers[0].hidden_ff.weight.data.cpu().detach().numpy())
e2 = np.linalg.eigvals(ssm.rnn_layers[1].hidden_ff.weight.data.cpu().detach().numpy())
plt.scatter(e1.real, e1.imag, s=1)
plt.scatter(e2.real, e2.imag, s=1)

In [ ]:
import numpy as np

e1_init = np.linalg.eigvals(ssm_init.rnn_layers[0].hidden_ff.weight.data.cpu().detach().numpy())
e2_init = np.linalg.eigvals(ssm_init.rnn_layers[1].hidden_ff.weight.data.cpu().detach().numpy())
plt.scatter(e1_init.real, e1_init.imag, s=1)
plt.scatter(e2_init.real, e2_init.imag, s=1)

In [ ]:
plt.scatter(e1_init.real, e1_init.imag, s=1)
plt.scatter(e1.real, e1.imag, s=1)